In [0]:
%run ./01_setup_environment

In [0]:


# ========================================
# 04_quarantine_validation
# ========================================

from pyspark.sql.functions import *

try:

    # ---------------------------------------
    # Read Bronze Patients Data
    # ---------------------------------------

    bronze_patients_df = spark.read.format("delta") \
        .load(f"{bronze_path}/patients")

    # ---------------------------------------
    # Invalid Records
    # ---------------------------------------

    invalid_patients_df = bronze_patients_df.filter(
        col("patient_id").isNull() |
        col("patient_name").isNull()
    )

    # ---------------------------------------
    # Valid Records
    # ---------------------------------------

    valid_patients_df = bronze_patients_df.filter(
        col("patient_id").isNotNull() &
        col("patient_name").isNotNull()
    )

    # ---------------------------------------
    # Write Invalid Records
    # ---------------------------------------

    invalid_patients_df.write \
        .format("delta") \
        .mode("append") \
        .save(f"{quarantine_path}/invalid_patients")

    # ---------------------------------------
    # Write Valid Records
    # ---------------------------------------

    valid_patients_df.write \
        .format("delta") \
        .mode("overwrite") \
        .save(f"{bronze_path}/validated_patients")

    # ---------------------------------------
    # Audit Logging
    # ---------------------------------------

    log_audit(
        "patients_validation_pipeline",
        "quarantine",
        "validated_patients",
        valid_patients_df.count(),
        "SUCCESS"
    )

    print("Quarantine Validation Completed Successfully")

except Exception as e:

    log_audit(
        "patients_validation_pipeline",
        "quarantine",
        "validated_patients",
        0,
        "FAILED",
        str(e)
    )

    raise e